# Notebook 16 - Scale Dynamics and Scale-Removal Ablations

This notebook runs two lightweight ablation studies for the Infinity-2B / VAR-style scale hypothesis.

**Study 1: generation step dynamics.** Randomly sample 50 content/style prompts, generate baseline images with the lightweight Infinity-2B GGUF backbone, decode the cumulative image after each AR scale, and measure how each intermediate step approaches the final output by RGB histogram distance, VGG content similarity, and DINO style similarity.

**Study 2: CSD100 scale-removal reconstruction.** Randomly sample 50 CSD100 images, encode each image into multi-scale VAE residuals, remove one scale at a time, reconstruct, and measure which scales are most style-sensitive.

The final cell saves all plot outputs and metric tables into a zip file for download.


In [ ]:
# Core experiment configuration.
from pathlib import Path
import gc
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile

SEED = 2026
random.seed(SEED)

ROOT = Path('/content/notebook_16_scale_ablation') if Path('/content').exists() else Path.cwd() / 'notebook_16_scale_ablation'
OFFICIAL_REPO = 'https://github.com/FoundationVision/Infinity.git'
GGUF_REPO = 'kzopp/Infinity-2B-GGUF_UNOFFICIAL'  # public mirror with the required GGUF files
OFFICIAL_DIR = ROOT / 'Infinity'
PORT_DIR = ROOT / 'gguf_port'
ASSET_DIR = ROOT / 'assets'
OUTPUT_DIR = ROOT / 'outputs'

STUDY1_DIR = OUTPUT_DIR / 'study1_step_dynamics'
STUDY1_TRACE_DIR = STUDY1_DIR / 'traces'
STUDY1_FINAL_DIR = STUDY1_DIR / 'final_images'
STUDY1_STEP_GRID_DIR = STUDY1_DIR / 'step_grids'
STUDY1_PLOT_DIR = STUDY1_DIR / 'plots'

STUDY2_DIR = OUTPUT_DIR / 'study2_scale_removal'
STUDY2_ORIGINAL_DIR = STUDY2_DIR / 'originals'
STUDY2_RECON_GRID_DIR = STUDY2_DIR / 'reconstruction_grids'
STUDY2_PLOT_DIR = STUDY2_DIR / 'plots'

for directory in [
    ROOT, PORT_DIR, ASSET_DIR, OUTPUT_DIR,
    STUDY1_TRACE_DIR, STUDY1_FINAL_DIR, STUDY1_STEP_GRID_DIR, STUDY1_PLOT_DIR,
    STUDY2_ORIGINAL_DIR, STUDY2_RECON_GRID_DIR, STUDY2_PLOT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

# Infinity-2B GGUF settings. 0.25M is the lightweight 512px preset.
MODEL_PN = '0.25M'
T5_DEVICE = 'cuda'
CFG_SCALE = 1.0
TAU = 0.1
TOP_K = 600
TOP_P = 0.95

STUDY1_NUM_PROMPTS = 50
STUDY2_NUM_IMAGES = 50
FORCE_REGENERATE_STUDY1 = False
FORCE_REGENERATE_STUDY2 = False

print('Root:', ROOT)
print('Outputs:', OUTPUT_DIR)


In [ ]:
# Runtime check.
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('A CUDA GPU runtime is required for practical Infinity-2B inference.')


In [ ]:
# Install dependencies. In Colab, run this once, then continue with the next cell.
packages = [
    'huggingface_hub', 'gguf', 'gradio', 'transformers', 'sentencepiece',
    'easydict', 'typed-argument-parser', 'seaborn', 'kornia', 'gputil',
    'colorama', 'omegaconf', 'timm==0.9.6', 'decord', 'pytz', 'imageio',
    'einops', 'opencv-python', 'accelerate', 'pandas', 'matplotlib', 'scikit-image'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependencies installed.')


In [ ]:
# Download the official Infinity source and the GGUF assets/loader.
# The GGUF files may require Hugging Face authentication. In Colab, add a secret
# named HF_TOKEN or HUGGINGFACE_HUB_TOKEN, then rerun this cell.
if not OFFICIAL_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', OFFICIAL_REPO, str(OFFICIAL_DIR)], check=True)
else:
    print('Official Infinity source already exists:', OFFICIAL_DIR)

from huggingface_hub import hf_hub_download, login
from huggingface_hub.errors import RepositoryNotFoundError, GatedRepoError, HfHubHTTPError


def get_hf_token():
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')
    if token:
        return token
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN') or userdata.get('HUGGINGFACE_HUB_TOKEN')
    except Exception:
        return None


HF_TOKEN = get_hf_token()
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Hugging Face token found and login completed.')
else:
    print('No Hugging Face token found. If the GGUF repo is private/gated, add HF_TOKEN in Colab Secrets.')


def download_hf_file(filename, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    try:
        return Path(hf_hub_download(
            repo_id=GGUF_REPO,
            filename=filename,
            local_dir=str(target_dir),
            token=HF_TOKEN,
        ))
    except (RepositoryNotFoundError, GatedRepoError, HfHubHTTPError) as exc:
        raise RuntimeError(
            f'Could not download {filename!r} from Hugging Face repo {GGUF_REPO!r}.\n'
            'This repo is currently not accessible anonymously from Colab. To fix it:\n'
            '1. Open Colab left sidebar > Secrets.\n'
            '2. Add a secret named HF_TOKEN with a Hugging Face token that has access to the repo.\n'
            '3. Enable notebook access to that secret, then rerun this cell.\n'
            'If you do not have access to the GGUF repo, use a runtime where these files already exist '
            'or replace GGUF_REPO / filenames with an accessible mirror.'
        ) from exc


PORT_SCRIPT = download_hf_file('generate_image_2b_q8_gguf.py', PORT_DIR)
PORT_UTILS = download_hf_file('infinity_gguf_utils.py', PORT_DIR)
PATCH_DIR = ROOT / 'gguf_patched_source'
PATCHED_BASIC = download_hf_file('Infinity/infinity/models/basic.py', PATCH_DIR)
PATCHED_INFINITY = download_hf_file('Infinity/infinity/models/infinity.py', PATCH_DIR)

# Copy GGUF-compatible model code into the official source tree.
official_basic = OFFICIAL_DIR / 'infinity' / 'models' / 'basic.py'
official_infinity = OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py'
shutil.copy2(PATCHED_BASIC, official_basic)
shutil.copy2(PATCHED_INFINITY, official_infinity)

# Patch the optional flash-attention guard for environments where flash_attn is absent.
infinity_source = official_infinity.read_text()
old_attention_guard = "customized_kernel_installed = any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
new_attention_guard = "customized_kernel_installed = flash_attn_func is not None and any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
if old_attention_guard in infinity_source:
    official_infinity.write_text(infinity_source.replace(old_attention_guard, new_attention_guard, 1))
else:
    print('Attention guard already patched or upstream source changed.')

INFINITY_GGUF = download_hf_file('infinity_2b_reg_Q8_0.gguf', ASSET_DIR)
T5_GGUF = download_hf_file('flan-t5-xl-encoder-Q8_0.gguf', ASSET_DIR)
VAE_PATH = download_hf_file('Infinity/infinity_vae_d32_reg.pth', ASSET_DIR)

for path in [PORT_SCRIPT, PORT_UTILS, PATCHED_BASIC, PATCHED_INFINITY, INFINITY_GGUF, T5_GGUF, VAE_PATH]:
    print(f'{path.name:45s} {path.stat().st_size / 2**30:.3f} GiB')


In [ ]:
# Memory-efficient T5 GGUF loader.
import numpy as np
import gguf

def load_t5_encoder_streaming(gguf_path, device='cpu'):
    from gguf import GGUFReader
    from transformers import T5Config, T5EncoderModel

    key_map = {
        'enc.': 'encoder.', '.blk.': '.block.', 'token_embd': 'shared',
        'output_norm': 'final_layer_norm', 'attn_q': 'layer.0.SelfAttention.q',
        'attn_k': 'layer.0.SelfAttention.k', 'attn_v': 'layer.0.SelfAttention.v',
        'attn_o': 'layer.0.SelfAttention.o', 'attn_norm': 'layer.0.layer_norm',
        'attn_rel_b': 'layer.0.SelfAttention.relative_attention_bias',
        'ffn_up': 'layer.1.DenseReluDense.wi_1', 'ffn_down': 'layer.1.DenseReluDense.wo',
        'ffn_gate': 'layer.1.DenseReluDense.wi_0', 'ffn_norm': 'layer.1.layer_norm',
    }

    config = T5Config.from_pretrained('google/flan-t5-xl')
    from accelerate import init_empty_weights
    with init_empty_weights():
        model = T5EncoderModel(config)
    model = model.to(dtype=torch.float16)
    model.to_empty(device=device)
    model.eval().requires_grad_(False)

    parameter_refs = dict(model.named_parameters())
    buffer_refs = dict(model.named_buffers())
    reader = GGUFReader(str(gguf_path))
    unquantized_types = {gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16}
    loaded, skipped = 0, []

    with torch.inference_mode():
        for tensor in reader.tensors:
            name = tensor.name
            for old_key, new_key in key_map.items():
                name = name.replace(old_key, new_key)
            shape = torch.Size(tuple(int(v) for v in reversed(tensor.shape)))
            raw = torch.from_numpy(np.array(tensor.data))
            if tensor.tensor_type not in unquantized_types:
                quant_param = gguf_loader.GGUFParameter(raw, quant_type=tensor.tensor_type)
                value = gguf_loader.dequantize_gguf_tensor(quant_param, target_dtype=torch.float16)
            else:
                value = raw.to(dtype=torch.float16)
            if value.numel() != math.prod(shape):
                skipped.append((name, 'numel mismatch'))
                del raw, value
                continue
            value = value.reshape(shape)
            target = parameter_refs.get(name)
            if target is None:
                target = buffer_refs.get(name)
            if target is None or tuple(target.shape) != tuple(shape):
                skipped.append((name, 'missing or shape mismatch'))
                del raw, value
                continue
            target.data.copy_(value.to(device=target.device, dtype=target.dtype))
            loaded += 1
            del raw, value

    del reader, parameter_refs, buffer_refs
    gc.collect()
    print(f'T5 tensors loaded: {loaded}; skipped: {len(skipped)}')
    if skipped:
        print('First skipped tensors:', skipped[:5])
    return model

print('Streaming T5 loader ready.')


In [ ]:
# Import the patched GGUF loader.
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, str(OFFICIAL_DIR))

loader_source = PORT_SCRIPT.read_text()
compat_pattern = r"\n    # Apply NumPy 2\.0 compatibility patch.*?\n    # Load GGUF state dict"
loader_source, replacements = re.subn(
    compat_pattern,
    '\n    # NumPy compatibility is handled by the installed gguf package.\n    # Load GGUF state dict',
    loader_source,
    count=1,
    flags=re.S,
)
print('Removed obsolete NumPy compatibility block:', replacements == 1)

PATCHED_LOADER = PORT_DIR / 'generate_image_2b_q8_gguf_colab.py'
PATCHED_LOADER.write_text(loader_source)
spec = importlib.util.spec_from_file_location('infinity_gguf_colab_loader', PATCHED_LOADER)
gguf_loader = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gguf_loader
spec.loader.exec_module(gguf_loader)
print('Custom GGUF loader imported successfully.')


In [ ]:
# Load tokenizer, T5 encoder, VAE, and Infinity-2B transformer.
print('[1/4] Loading T5 tokenizer...')
text_tokenizer = gguf_loader.load_t5_tokenizer_from_gguf(str(T5_GGUF))

print(f'[2/4] Streaming quantized T5 encoder to {T5_DEVICE}...')
text_encoder = load_t5_encoder_streaming(str(T5_GGUF), device=T5_DEVICE)

print('[3/4] Loading VAE on GPU...')
vae = gguf_loader.load_vae(str(VAE_PATH), vae_type=32, device=DEVICE)

print('[4/4] Loading quantized Infinity-2B transformer on GPU...')
infinity_model = gguf_loader.load_infinity_from_gguf(
    str(INFINITY_GGUF),
    vae=vae,
    device=DEVICE,
    model_type='infinity_2b',
    text_channels=2048,
    pn=MODEL_PN,
)

infinity_model.eval()
vae.eval()
print('All generation components loaded successfully.')


In [ ]:
# Build the official dynamic-resolution schedule and define image helpers.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
import torchvision
from PIL import Image, ImageOps, ImageDraw
from tqdm.auto import tqdm
from infinity.utils.dynamic_resolution import dynamic_resolution_h_w, h_div_w_templates
from infinity.models.infinity import sample_with_top_k_top_p_also_inplace_modifying_logits_

ASPECT_RATIO = 1.0
h_div_w_template = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - ASPECT_RATIO))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template][MODEL_PN]['scales']
SCALE_SCHEDULE = [(1, h, w) for (_, h, w) in scale_schedule]
PATCH_NUMS = tuple(h for (_, h, w) in SCALE_SCHEDULE)
IMAGE_SIZE_HW = (1024, 1024) if MODEL_PN == '1M' else (512, 512)
device = DEVICE
infinity = infinity_model

if not hasattr(vae.quantizer, 'lfq'):
    vae.quantizer.lfq = vae.quantizer.bsq

print('Scale schedule:', SCALE_SCHEDULE)
print('Patch counts:', PATCH_NUMS)
print('Image size:', IMAGE_SIZE_HW)


def tensor_to_pil(image):
    if isinstance(image, (list, tuple)):
        image = image[0]
    tensor = image.detach().float().cpu() if torch.is_tensor(image) else torch.as_tensor(image).float()
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise ValueError(f'Unexpected image shape: {tuple(tensor.shape)}')
    if tensor.shape[0] in (1, 3, 4):
        tensor = tensor.permute(1, 2, 0)
    if tensor.shape[-1] == 1:
        tensor = tensor.repeat(1, 1, 3)
    if tensor.shape[-1] > 3:
        tensor = tensor[..., :3]
    lo, hi = float(tensor.min()), float(tensor.max())
    if lo < -0.05:
        tensor = (tensor + 1.0) / 2.0
    elif hi > 1.05:
        tensor = tensor / 255.0
    array = (tensor.clamp(0, 1).numpy() * 255).round().astype('uint8')
    return Image.fromarray(array, mode='RGB')


def save_image_tensor(image_01, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tensor_to_pil(image_01).save(path)
    return path


## Study 1 - Step-Wise Generation Dynamics

This section generates 50 baseline images and stores the cumulative VAE code after every autoregressive scale. Later cells decode those per-scale snapshots and compare every intermediate output to the final output.


In [ ]:
# Prompt set: deterministic random sample of 50 content/style combinations.
OBJECTS = [
    'cat', 'fox', 'robot', 'spaceship', 'mushroom', 'teapot', 'violin', 'camera', 'lantern', 'penguin',
    'castle', 'dragon', 'sailboat', 'train', 'butterfly', 'flower', 'owl', 'bicycle', 'chair', 'lamp',
    'rocket', 'turtle', 'parrot', 'watch', 'piano', 'watermelon', 'umbrella', 'balloon', 'horse', 'seashell',
    'snail', 'duck', 'bear', 'notebook', 'paintbrush', 'saxophone', 'crown', 'compass', 'microphone', 'orchid',
    'frisbee', 'leopard', 'glass', 'lollipop', 'hammer', 'kangaroo', 'ladybug', 'moose', 'ninja', 'seagull'
]
SUPERCLASSES = {
    'cat': 'animals', 'fox': 'animals', 'robot': 'toys', 'spaceship': 'vehicles', 'mushroom': 'plants',
    'teapot': 'objects', 'violin': 'musical instruments', 'camera': 'objects', 'lantern': 'objects',
    'penguin': 'animals', 'castle': 'buildings', 'dragon': 'fantasy creatures', 'sailboat': 'vehicles',
    'train': 'vehicles', 'butterfly': 'animals', 'flower': 'plants', 'owl': 'animals', 'bicycle': 'vehicles',
    'chair': 'furniture', 'lamp': 'objects', 'rocket': 'vehicles', 'turtle': 'animals', 'parrot': 'animals',
    'watch': 'objects', 'piano': 'musical instruments', 'watermelon': 'food', 'umbrella': 'objects',
    'balloon': 'objects', 'horse': 'animals', 'seashell': 'natural objects', 'snail': 'animals',
    'duck': 'animals', 'bear': 'animals', 'notebook': 'objects', 'paintbrush': 'objects',
    'saxophone': 'musical instruments', 'crown': 'objects', 'compass': 'objects', 'microphone': 'objects',
    'orchid': 'plants', 'frisbee': 'objects', 'leopard': 'animals', 'glass': 'objects', 'lollipop': 'food',
    'hammer': 'tools', 'kangaroo': 'animals', 'ladybug': 'animals', 'moose': 'animals', 'ninja': 'characters',
    'seagull': 'animals'
}
STYLES = [
    'watercolor painting', 'glowing neon', 'retro comic book', 'origami paper craft', 'blueprint drawing',
    'minimal pastel illustration', 'mosaic tile art', 'pixel art', 'woodcut print', 'chalk drawing',
    'digital glitch art', 'cubist painting', 'graffiti mural', 'papercut collage', 'sticker illustration',
    'impressionist painting', 'surrealist painting', 'flat vector illustration', 'medieval fantasy illustration',
    'rainbow flowing smoke wave'
]
SETTINGS = [
    'centered composition', 'studio lighting', 'dark cinematic background', 'simple clean background',
    'floating in space', 'on a wooden table', 'in a quiet forest', 'beside a reflective lake',
    'under warm sunset light', 'on a white museum pedestal'
]

rng = random.Random(SEED)
combinations = []
for obj in OBJECTS:
    style = rng.choice(STYLES)
    setting = rng.choice(SETTINGS)
    superclass = SUPERCLASSES.get(obj, 'objects')
    prompt = f'A {obj}, {superclass}, in {style} style, {setting}'
    combinations.append({'case_id': len(combinations), 'object': obj, 'superclass': superclass, 'style': style, 'setting': setting, 'prompt': prompt})

PROMPT_ROWS = combinations[:STUDY1_NUM_PROMPTS]
prompt_df = pd.DataFrame(PROMPT_ROWS)
prompt_manifest_path = STUDY1_DIR / 'study1_prompts_50.csv'
prompt_df.to_csv(prompt_manifest_path, index=False)
prompt_df.head()


In [ ]:
# Baseline Infinity generation with a cumulative code trace after every scale.
def encode_prompts(prompts):
    if isinstance(prompts, str):
        prompts = [prompts]
    tokens = text_tokenizer(text=list(prompts), max_length=512, padding='max_length', truncation=True, return_tensors='pt')
    input_ids = tokens.input_ids.to(device, non_blocking=True)
    mask = tokens.attention_mask.to(device, non_blocking=True)
    with torch.no_grad():
        text_features = text_encoder(input_ids=input_ids, attention_mask=mask)['last_hidden_state'].float()
    lens = mask.sum(dim=-1).tolist()
    cu_seqlens_k = F.pad(mask.sum(dim=-1).to(dtype=torch.int32).cumsum_(0), (1, 0))
    max_seqlen_k = max(lens)
    kv_compact = torch.cat([feat_i[:len_i] for len_i, feat_i in zip(lens, text_features.unbind(0))], dim=0)
    return kv_compact, lens, cu_seqlens_k, max_seqlen_k


def _sample_bit_labels(logits_bl2d, rng, top_k=TOP_K, top_p=TOP_P):
    batch, seq_len = logits_bl2d.shape[:2]
    logits = logits_bl2d.reshape(batch, -1, 2).clone()
    sampled = sample_with_top_k_top_p_also_inplace_modifying_logits_(
        logits, rng=rng, top_k=top_k, top_p=top_p, num_samples=1
    )[:, :, 0]
    return sampled.reshape(batch, seq_len, -1)


def _bit_labels_to_codes(idx_bld, pn):
    idx = idx_bld.reshape(idx_bld.shape[0], pn[1], pn[2], -1).unsqueeze(1)
    return vae.quantizer.lfq.indices_to_codes(idx, label_type='bit_label')


def _next_raw_from_summed_codes(summed_codes, next_scale):
    last_stage = F.interpolate(summed_codes, size=next_scale, mode=vae.quantizer.z_interplote_up)
    last_stage = last_stage.squeeze(-3)
    if infinity.apply_spatial_patchify:
        last_stage = F.pixel_unshuffle(last_stage, 2)
    last_stage = last_stage.reshape(*last_stage.shape[:2], -1).permute(0, 2, 1)
    return last_stage


def _decode_summed_codes_to_image_01(summed_codes):
    image = vae.decode(summed_codes.squeeze(-3))
    return image.add(1).mul(0.5).clamp(0, 1)


@torch.no_grad()
def generate_text_trace(prompt, seed=SEED, cfg=CFG_SCALE, tau=TAU, top_k=TOP_K, top_p=TOP_P):
    model = infinity
    model.eval()
    rng = torch.Generator(device=device).manual_seed(int(seed))

    kv_compact, lens, cu_seqlens_k, max_seqlen_k = encode_prompts([prompt])
    kv_compact_un = kv_compact.clone()
    kv_compact_un[:lens[0]] = model.cfg_uncond[:lens[0]]
    kv_compact = torch.cat((kv_compact, kv_compact_un), dim=0)
    cu_seqlens_k = torch.cat((cu_seqlens_k, cu_seqlens_k[1:] + cu_seqlens_k[-1]), dim=0)
    batch_size = 2

    kv_compact = model.text_norm(kv_compact)
    sos = cond_BD = model.text_proj_for_sos((kv_compact, cu_seqlens_k, max_seqlen_k))
    kv_compact = model.text_proj_for_ca(kv_compact)
    ca_kv = kv_compact, cu_seqlens_k, max_seqlen_k
    last_stage = sos.unsqueeze(1).expand(batch_size, 1, -1) + model.pos_start.expand(batch_size, 1, -1)

    with torch.amp.autocast('cuda', enabled=False):
        cond_BD_or_gss = model.shared_ada_lin(cond_BD.float()).float().contiguous()

    final_size = SCALE_SCHEDULE[-1]
    summed_codes = last_stage.new_zeros(1, model.d_vae, *final_size)
    residuals, cumulative_trace = [], []

    for block in model.unregistered_blocks:
        block.sa.kv_caching(True)

    try:
        with torch.amp.autocast('cuda', enabled=True, dtype=torch.bfloat16, cache_enabled=True):
            for step_id, pn in enumerate(SCALE_SCHEDULE):
                need_to_pad = 0
                attn_fn = None
                if model.use_flex_attn:
                    attn_fn = model.attn_fn_compile_dict.get(tuple(SCALE_SCHEDULE[:step_id + 1]), None)

                for block_idx, block_chunk in enumerate(model.block_chunks):
                    if model.add_lvl_embeding_only_first_block and block_idx == 0:
                        last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)
                    if not model.add_lvl_embeding_only_first_block:
                        last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)

                    for block in block_chunk.module:
                        last_stage = block(
                            x=last_stage,
                            cond_BD=cond_BD_or_gss,
                            ca_kv=ca_kv,
                            attn_bias_or_two_vector=None,
                            attn_fn=attn_fn,
                            scale_schedule=SCALE_SCHEDULE,
                            rope2d_freqs_grid=model.rope2d_freqs_grid,
                            scale_ind=step_id,
                        )

                logits = model.get_logits(last_stage, cond_BD).mul(1 / float(tau))
                logits = float(cfg) * logits[:1] + (1 - float(cfg)) * logits[1:]
                idx = _sample_bit_labels(logits, rng, top_k=top_k, top_p=top_p)
                codes = _bit_labels_to_codes(idx, pn)
                if step_id != len(SCALE_SCHEDULE) - 1:
                    codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)

                residuals.append(codes.detach().float().cpu())
                summed_codes = summed_codes + codes
                cumulative_trace.append(summed_codes.detach().float().cpu())

                if step_id != len(SCALE_SCHEDULE) - 1:
                    next_raw = _next_raw_from_summed_codes(summed_codes, SCALE_SCHEDULE[step_id + 1])
                    last_stage = model.word_embed(model.norm0_ve(next_raw)).repeat(batch_size, 1, 1)

        final_image_01 = _decode_summed_codes_to_image_01(summed_codes)
        return {
            'prompt': prompt,
            'seed': int(seed),
            'final_image_01': final_image_01.detach().float().cpu(),
            'trace': cumulative_trace,
            'residuals': residuals,
        }
    finally:
        for block in model.unregistered_blocks:
            block.sa.kv_caching(False)


In [ ]:
# Run Study 1 generation. This can take a while for 50 prompts.
def study1_trace_path(case_id):
    return STUDY1_TRACE_DIR / f'case_{int(case_id):03d}_trace.pt'

def study1_final_path(case_id):
    return STUDY1_FINAL_DIR / f'case_{int(case_id):03d}.png'

RUN_STUDY1_GENERATION = True

if RUN_STUDY1_GENERATION:
    for row in tqdm(PROMPT_ROWS, desc='Study 1: generating traced images'):
        case_id = int(row['case_id'])
        trace_path = study1_trace_path(case_id)
        final_path = study1_final_path(case_id)
        if trace_path.exists() and final_path.exists() and not FORCE_REGENERATE_STUDY1:
            continue
        result = generate_text_trace(row['prompt'], seed=SEED + case_id)
        save_image_tensor(result['final_image_01'], final_path)
        torch.save({
            'case_id': case_id,
            'prompt': row['prompt'],
            'seed': SEED + case_id,
            'scale_schedule': SCALE_SCHEDULE,
            'trace': result['trace'],
            'residuals': result['residuals'],
        }, trace_path)
        del result
        gc.collect()
        torch.cuda.empty_cache()
    print('Study 1 generation/traces are ready:', STUDY1_TRACE_DIR)
else:
    print('Set RUN_STUDY1_GENERATION = True to generate traces.')


In [ ]:
# Metric backbones and similarity utilities.
from torchvision.models import vgg19, VGG19_Weights, resnet50, ResNet50_Weights

METRIC_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
metric_tf = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

vgg_weights = VGG19_Weights.IMAGENET1K_V1
vgg = vgg19(weights=vgg_weights).features.to(METRIC_DEVICE).eval()
for parameter in vgg.parameters():
    parameter.requires_grad_(False)

try:
    style_backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(METRIC_DEVICE).eval()
    STYLE_BACKBONE_NAME = 'DINOv2 ViT-S/14'
except Exception as exc:
    print('DINOv2 download/load failed, falling back to ResNet50 style embedding:', repr(exc))
    resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).to(METRIC_DEVICE).eval()
    style_backbone = torch.nn.Sequential(*(list(resnet.children())[:-1])).to(METRIC_DEVICE).eval()
    STYLE_BACKBONE_NAME = 'ResNet50 fallback'
for parameter in style_backbone.parameters():
    parameter.requires_grad_(False)

print('Content metric: VGG19 features')
print('Style metric:', STYLE_BACKBONE_NAME)


def image_for_metrics(image):
    if torch.is_tensor(image):
        pil = tensor_to_pil(image)
    elif isinstance(image, Image.Image):
        pil = image.convert('RGB')
    else:
        pil = Image.open(image).convert('RGB')
    return pil


def metric_input(image):
    return metric_tf(image_for_metrics(image)).unsqueeze(0).to(METRIC_DEVICE)


@torch.no_grad()
def vgg_content_embedding(image):
    x = metric_input(image)
    for i, layer in enumerate(vgg):
        x = layer(x)
        if i == 35:  # relu5_4 neighborhood
            break
    x = F.adaptive_avg_pool2d(x.float(), output_size=1).flatten(1)
    return F.normalize(x, dim=-1)


@torch.no_grad()
def dino_style_embedding(image):
    x = metric_input(image)
    feat = style_backbone(x)
    if isinstance(feat, dict):
        feat = feat.get('x_norm_clstoken', None)
        if feat is None:
            feat = feat.get('x_prenorm', None)
        if feat is None:
            raise KeyError('Could not find a usable DINO feature tensor in the model output dictionary.')
    feat = feat.float().flatten(1)
    return F.normalize(feat, dim=-1)


@torch.no_grad()
def vgg_gram_style_embedding(image):
    x = metric_input(image)
    grams = []
    target_layers = {3, 8, 17, 26}
    for i, layer in enumerate(vgg):
        x = layer(x)
        if i in target_layers:
            b, c, h, w = x.shape
            features = x.float().reshape(b, c, h * w)
            gram = torch.bmm(features, features.transpose(1, 2)) / max(c * h * w, 1)
            grams.append(F.normalize(gram.flatten(1), dim=-1))
    return F.normalize(torch.cat(grams, dim=-1), dim=-1)


def cosine01(a, b):
    return float((a * b).sum(dim=-1).clamp(-1, 1).detach().cpu().item())


def rgb_chi_square_distance(image_a, image_b, bins=32, eps=1e-8):
    a = np.asarray(image_for_metrics(image_a).resize((224, 224)), dtype=np.float32) / 255.0
    b = np.asarray(image_for_metrics(image_b).resize((224, 224)), dtype=np.float32) / 255.0
    distances = []
    for channel in range(3):
        hist_a, _ = np.histogram(a[..., channel], bins=bins, range=(0, 1), density=False)
        hist_b, _ = np.histogram(b[..., channel], bins=bins, range=(0, 1), density=False)
        hist_a = hist_a.astype(np.float64) / max(hist_a.sum(), 1)
        hist_b = hist_b.astype(np.float64) / max(hist_b.sum(), 1)
        distances.append(0.5 * np.sum(((hist_a - hist_b) ** 2) / (hist_a + hist_b + eps)))
    return float(np.mean(distances))


In [ ]:
# Decode Study 1 traces, save 50 step-grid visualizations, and compute per-step metrics.
def make_step_grid(images, title, path, columns=None):
    columns = columns or len(images)
    thumb = 160
    label_h = 28
    rows = math.ceil(len(images) / columns)
    canvas = Image.new('RGB', (columns * thumb, rows * (thumb + label_h) + 34), 'white')
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), title[:180], fill=(0, 0, 0))
    for idx, image in enumerate(images):
        row, col = divmod(idx, columns)
        x = col * thumb
        y = 34 + row * (thumb + label_h)
        canvas.paste(image.resize((thumb, thumb), Image.Resampling.LANCZOS), (x, y))
        draw.text((x + 6, y + thumb + 6), f'scale {idx + 1}', fill=(0, 0, 0))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path)
    return path


@torch.no_grad()
def decode_trace_payload(payload):
    images = []
    for summed in payload['trace']:
        image_01 = _decode_summed_codes_to_image_01(summed.to(device))
        images.append(tensor_to_pil(image_01))
        del image_01
    return images


RUN_STUDY1_METRICS = True
study1_metrics_path = STUDY1_DIR / 'study1_stepwise_metrics.csv'

if RUN_STUDY1_METRICS:
    rows = []
    for row in tqdm(PROMPT_ROWS, desc='Study 1: metrics'):
        case_id = int(row['case_id'])
        payload = torch.load(study1_trace_path(case_id), map_location='cpu')
        step_images = decode_trace_payload(payload)
        final_image = step_images[-1]
        make_step_grid(step_images, f'case {case_id:03d}: {row["prompt"]}', STUDY1_STEP_GRID_DIR / f'case_{case_id:03d}_steps.png')

        final_content = vgg_content_embedding(final_image)
        final_style = dino_style_embedding(final_image)
        for step_idx, step_image in enumerate(step_images, start=1):
            rows.append({
                'case_id': case_id,
                'prompt': row['prompt'],
                'step': step_idx,
                'rgb_chi_square': rgb_chi_square_distance(step_image, final_image),
                'content_similarity': cosine01(vgg_content_embedding(step_image), final_content),
                'style_similarity': cosine01(dino_style_embedding(step_image), final_style),
            })
        del payload, step_images, final_content, final_style
        gc.collect()
        torch.cuda.empty_cache()

    study1_metrics_df = pd.DataFrame(rows)
    study1_metrics_df.to_csv(study1_metrics_path, index=False)
    print('Saved Study 1 metrics:', study1_metrics_path)
else:
    study1_metrics_df = pd.read_csv(study1_metrics_path)
    print('Loaded Study 1 metrics:', study1_metrics_path)

study1_metrics_df.head()


In [ ]:
# Plot Study 1: 50 faint per-prompt lines plus the mean trajectory.
study1_metrics_df = pd.read_csv(study1_metrics_path)
summary1 = study1_metrics_df.groupby('step', as_index=False).agg(
    rgb_chi_square_mean=('rgb_chi_square', 'mean'),
    rgb_chi_square_std=('rgb_chi_square', 'std'),
    content_similarity_mean=('content_similarity', 'mean'),
    content_similarity_std=('content_similarity', 'std'),
    style_similarity_mean=('style_similarity', 'mean'),
    style_similarity_std=('style_similarity', 'std'),
)
summary1_path = STUDY1_DIR / 'study1_stepwise_summary.csv'
summary1.to_csv(summary1_path, index=False)

sns.set_theme(style='whitegrid', context='paper')
metrics = [
    ('rgb_chi_square', 'RGB histogram distance to final', '#ff7f0e'),
    ('content_similarity', 'VGG content similarity to final', '#bcbd22'),
    ('style_similarity', f'{STYLE_BACKBONE_NAME} style similarity to final', '#556b2f'),
]
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
for ax, (column, ylabel, color) in zip(axes, metrics):
    for _, group in study1_metrics_df.groupby('case_id'):
        ax.plot(group['step'], group[column], color=color, alpha=0.12, linewidth=0.8)
    mean = study1_metrics_df.groupby('step')[column].mean()
    std = study1_metrics_df.groupby('step')[column].std()
    steps = mean.index.to_numpy()
    ax.plot(steps, mean.to_numpy(), color=color, linewidth=2.8, marker='o', label='mean over 50 prompts')
    ax.fill_between(steps, mean.to_numpy() - std.to_numpy(), mean.to_numpy() + std.to_numpy(), color=color, alpha=0.16, linewidth=0)
    ax.set_ylabel(ylabel)
    ax.legend(loc='best')
axes[-1].set_xlabel('Generation scale / step')
fig.suptitle('Study 1: step-wise baseline generation dynamics', y=1.0, fontsize=14, fontweight='bold')
fig.tight_layout()
study1_plot_path = STUDY1_PLOT_DIR / 'study1_stepwise_dynamics_50_lines.png'
fig.savefig(study1_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', study1_plot_path)

# Compact paper-style numerical plot with mean curves only.
fig, ax1 = plt.subplots(figsize=(10, 4.8))
ax2 = ax1.twinx()
ax1.plot(summary1['step'], summary1['rgb_chi_square_mean'] * 1e4, color='#ff7f0e', marker='*', linewidth=2.4, markersize=9, label='RGB statistics (x1e4)')
ax2.plot(summary1['step'], summary1['content_similarity_mean'], color='#bcbd22', marker='o', linewidth=2.4, label='Content similarity')
ax2.plot(summary1['step'], summary1['style_similarity_mean'], color='#556b2f', marker='D', linewidth=2.4, linestyle=':', label='Style similarity')
ax1.set_xlabel('Number of steps / scale')
ax1.set_ylabel('RGB histogram distance x 1e4', color='#ff7f0e')
ax2.set_ylabel('Similarity to final image')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='lower right')
ax1.grid(True, linestyle='--', alpha=0.35)
fig.tight_layout()
study1_mean_plot_path = STUDY1_PLOT_DIR / 'study1_stepwise_dynamics_mean_curves.png'
fig.savefig(study1_mean_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', study1_mean_plot_path)
print('Saved summary:', summary1_path)


## Study 2 - CSD100 Scale-Removal Reconstruction

This section mirrors the CSD-VAR scale-sensitivity diagnostic: encode real CSD100 images, remove one scale at a time, reconstruct, and score how much the missing scale changes the reconstruction. Lower similarity after removing a scale means that scale was more important.


In [ ]:
# Locate or clone the CSD100 folder.
WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
local_workspace = Path('/Users/builehoang/Documents/Projects/Image Generation/VAR_Style_Transfer_Workspace')
clean_workspace = Path('/content/VAR_Style_Transfer_Workspace') if Path('/content').exists() else ROOT / 'VAR_Style_Transfer_Workspace'

WORKSPACE = local_workspace if local_workspace.exists() else clean_workspace
if not WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', WORKSPACE_REPO, str(WORKSPACE)], check=True)

CSD100_DIR = WORKSPACE / 'csd100'
image_paths = sorted(CSD100_DIR.glob('*/00.jpg'))
if len(image_paths) < STUDY2_NUM_IMAGES:
    print(f'Found only {len(image_paths)} CSD100 images in {CSD100_DIR}. Using a clean clone under {clean_workspace}.')
    if not clean_workspace.exists():
        subprocess.run(['git', 'clone', '--depth', '1', WORKSPACE_REPO, str(clean_workspace)], check=True)
    WORKSPACE = clean_workspace
    CSD100_DIR = WORKSPACE / 'csd100'
    image_paths = sorted(CSD100_DIR.glob('*/00.jpg'))

if len(image_paths) < STUDY2_NUM_IMAGES:
    raise FileNotFoundError(f'Need at least {STUDY2_NUM_IMAGES} CSD100 images, found {len(image_paths)} in {CSD100_DIR}')

rng = random.Random(SEED)
sampled_paths = rng.sample(image_paths, STUDY2_NUM_IMAGES)
study2_manifest = pd.DataFrame([
    {'case_id': i, 'image_path': str(path), 'pair_id': path.parent.name}
    for i, path in enumerate(sampled_paths)
])
study2_manifest_path = STUDY2_DIR / 'study2_csd100_sample_50.csv'
study2_manifest.to_csv(study2_manifest_path, index=False)
print('CSD100 folder:', CSD100_DIR)
print('Saved manifest:', study2_manifest_path)
study2_manifest.head()


In [ ]:
# Encode CSD100 images into multi-scale VAE residuals and reconstruct with one scale removed.
def load_image_m11(path, size=512):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    tensor_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    return tensor_01.mul(2).sub(1), tensor_01, image


@torch.no_grad()
def encode_image_residuals(image_m11):
    with torch.amp.autocast('cuda', enabled=False):
        _, _, _, all_bit_indices, _, _ = vae.encode(image_m11.float(), scale_schedule=SCALE_SCHEDULE)
    residuals = []
    final_size = SCALE_SCHEDULE[-1]
    for step_id, bit_indices in enumerate(all_bit_indices):
        codes = vae.quantizer.lfq.indices_to_codes(bit_indices, label_type='bit_label')
        if step_id != len(SCALE_SCHEDULE) - 1:
            codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)
        residuals.append(codes.detach().float().cpu())
    return residuals


@torch.no_grad()
def reconstruct_from_residuals(residuals, remove_index=None):
    kept = [residual.to(device) for i, residual in enumerate(residuals) if remove_index is None or i != remove_index]
    if not kept:
        raise ValueError('At least one residual scale must remain for reconstruction.')
    summed = torch.stack(kept, dim=0).sum(dim=0)
    image_01 = _decode_summed_codes_to_image_01(summed)
    return tensor_to_pil(image_01)


def make_reconstruction_grid(original_image, recon_images, title, path):
    labels = ['original'] + [f'-scale {i + 1}' for i in range(len(recon_images))]
    images = [original_image] + recon_images
    thumb = 128
    label_h = 24
    cols = min(6, len(images))
    rows = math.ceil(len(images) / cols)
    canvas = Image.new('RGB', (cols * thumb, rows * (thumb + label_h) + 34), 'white')
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), title[:160], fill=(0, 0, 0))
    for idx, (image, label) in enumerate(zip(images, labels)):
        row, col = divmod(idx, cols)
        x = col * thumb
        y = 34 + row * (thumb + label_h)
        canvas.paste(image.resize((thumb, thumb), Image.Resampling.LANCZOS), (x, y))
        draw.text((x + 5, y + thumb + 5), label, fill=(0, 0, 0))
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path)
    return path


In [ ]:
# Run Study 2 and compute DINO/content-style proxy similarities.
study2_metrics_path = STUDY2_DIR / 'study2_scale_removal_metrics.csv'

RUN_STUDY2 = True
if RUN_STUDY2:
    metric_rows = []
    for row in tqdm(study2_manifest.to_dict('records'), desc='Study 2: CSD100 scale removal'):
        case_id = int(row['case_id'])
        original_m11, original_01, original_image = load_image_m11(row['image_path'], size=IMAGE_SIZE_HW[0])
        original_save_path = STUDY2_ORIGINAL_DIR / f'case_{case_id:03d}_{row["pair_id"]}.png'
        if FORCE_REGENERATE_STUDY2 or not original_save_path.exists():
            original_image.save(original_save_path)

        residuals = encode_image_residuals(original_m11)
        original_dino = dino_style_embedding(original_image)
        original_style_proxy = vgg_gram_style_embedding(original_image)
        original_content = vgg_content_embedding(original_image)

        recon_images = []
        for remove_index in range(len(residuals)):
            recon_image = reconstruct_from_residuals(residuals, remove_index=remove_index)
            recon_images.append(recon_image)
            metric_rows.append({
                'case_id': case_id,
                'pair_id': row['pair_id'],
                'image_path': row['image_path'],
                'removed_scale': remove_index + 1,
                'dino_similarity': cosine01(dino_style_embedding(recon_image), original_dino),
                'style_proxy_similarity': cosine01(vgg_gram_style_embedding(recon_image), original_style_proxy),
                'content_similarity': cosine01(vgg_content_embedding(recon_image), original_content),
            })

        make_reconstruction_grid(
            original_image,
            recon_images,
            f'case {case_id:03d}: {row["pair_id"]}',
            STUDY2_RECON_GRID_DIR / f'case_{case_id:03d}_{row["pair_id"]}_remove_each_scale.png',
        )
        del residuals, recon_images, original_m11, original_01, original_dino, original_style_proxy, original_content
        gc.collect()
        torch.cuda.empty_cache()

    study2_metrics_df = pd.DataFrame(metric_rows)
    study2_metrics_df.to_csv(study2_metrics_path, index=False)
    print('Saved Study 2 metrics:', study2_metrics_path)
else:
    study2_metrics_df = pd.read_csv(study2_metrics_path)
    print('Loaded Study 2 metrics:', study2_metrics_path)

study2_metrics_df.head()


In [ ]:
# Plot Study 2: scale sensitivity curves.
study2_metrics_df = pd.read_csv(study2_metrics_path)
summary2 = study2_metrics_df.groupby('removed_scale', as_index=False).agg(
    dino_mean=('dino_similarity', 'mean'),
    dino_std=('dino_similarity', 'std'),
    style_proxy_mean=('style_proxy_similarity', 'mean'),
    style_proxy_std=('style_proxy_similarity', 'std'),
    content_mean=('content_similarity', 'mean'),
    content_std=('content_similarity', 'std'),
)
summary2_path = STUDY2_DIR / 'study2_scale_removal_summary.csv'
summary2.to_csv(summary2_path, index=False)

fig, ax = plt.subplots(figsize=(8.5, 5.0))
for column, std_column, label, color, marker in [
    ('dino_mean', 'dino_std', 'DINO', '#ff6666', 'o'),
    ('style_proxy_mean', 'style_proxy_std', 'VGG Gram style proxy', '#4ecdc4', 's'),
    ('content_mean', 'content_std', 'VGG content', '#8d6eec', '^'),
]:
    x = summary2['removed_scale'].to_numpy()
    y = summary2[column].to_numpy()
    ystd = summary2[std_column].fillna(0).to_numpy()
    ax.plot(x, y, marker=marker, linewidth=2.4, markersize=7, label=label, color=color)
    ax.fill_between(x, y - ystd, y + ystd, color=color, alpha=0.12, linewidth=0)

ax.set_xlabel('Removed scale')
ax.set_ylabel('Similarity to original image')
ax.set_title('Study 2: CSD100 scale-removal reconstruction sensitivity', fontweight='bold')
ax.set_xticks(summary2['removed_scale'])
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
study2_plot_path = STUDY2_PLOT_DIR / 'study2_scale_removal_similarity.png'
fig.savefig(study2_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', study2_plot_path)
print('Saved summary:', summary2_path)

# Style-focused version matching the paper chart more closely.
fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.plot(summary2['removed_scale'], summary2['dino_mean'], marker='o', linewidth=2.4, markersize=7, label='DINO', color='#ff6666')
ax.plot(summary2['removed_scale'], summary2['style_proxy_mean'], marker='s', linewidth=2.4, markersize=7, label='Style proxy', color='#4ecdc4')
ax.set_xlabel('Removing Scale')
ax.set_ylabel('Score')
ax.set_title('Analysis of style-related scores across different scales', fontweight='bold')
ax.set_xticks(summary2['removed_scale'])
ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
study2_style_plot_path = STUDY2_PLOT_DIR / 'study2_scale_removal_style_scores.png'
fig.savefig(study2_style_plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', study2_style_plot_path)


In [ ]:
# Final cell: save all plot outputs and metric tables into one downloadable zip.
def package_notebook16_outputs(include_step_grids=True, include_trace_tensors=False):
    package_path = OUTPUT_DIR / 'notebook16_ablation_outputs.zip'
    suffixes = {'.png', '.jpg', '.jpeg', '.csv', '.json'}
    if include_trace_tensors:
        suffixes.add('.pt')

    with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(OUTPUT_DIR.rglob('*')):
            if path == package_path or not path.is_file():
                continue
            if path.suffix.lower() not in suffixes:
                continue
            if not include_step_grids and ('step_grids' in path.parts or 'reconstruction_grids' in path.parts):
                continue
            zf.write(path, path.relative_to(OUTPUT_DIR))

    manifest = {
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'output_root': str(OUTPUT_DIR),
        'zip_path': str(package_path),
        'study1_metrics': str(study1_metrics_path),
        'study2_metrics': str(study2_metrics_path),
        'style_metric_backbone': STYLE_BACKBONE_NAME,
        'scale_schedule': SCALE_SCHEDULE,
    }
    manifest_path = OUTPUT_DIR / 'notebook16_manifest.json'
    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(json.dumps(manifest, indent=2))
    print(f'Packaged outputs: {package_path} ({package_path.stat().st_size / 2**20:.2f} MiB)')

    if Path('/content').exists():
        try:
            from google.colab import files
            files.download(str(package_path))
        except Exception as exc:
            print('Automatic Colab download failed:', repr(exc))
            print('Download manually from:', package_path)
    return package_path

ZIP_PATH = package_notebook16_outputs(include_step_grids=True, include_trace_tensors=False)
